# Apache Kafka Deep Dive for Data Engineers
## Architecture, Producers, Consumers & Production Patterns

**What you'll learn:**
- Kafka architecture: brokers, topics, partitions, replicas
- Producer API: partitioning strategies, acknowledgements, idempotence
- Consumer API: consumer groups, offset management, rebalancing
- Delivery guarantees: at-most-once, at-least-once, exactly-once
- Schema Registry and schema evolution (Avro, Protobuf)
- Kafka Connect: source and sink connectors
- Kafka Streams vs Flink vs Spark Streaming comparison
- Operations: monitoring, retention, compaction, replication
- CDC with Debezium + Kafka
- Kafka + Spark Structured Streaming integration
- Production patterns and anti-patterns
- Interview questions

**Prerequisites:** 10_streaming_processing.ipynb, 48_hadoop_ecosystem.ipynb
**Estimated Time:** 7-9 hours

---

> Kafka is THE backbone of real-time data infrastructure. Every major tech company
> (LinkedIn, Uber, Netflix, Airbnb, Spotify) runs Kafka at the center of their
> data platform. If you do data engineering, you WILL work with Kafka.

---

---
# Section 1: Kafka Architecture — The Complete Picture

## Why Kafka Exists

Traditional messaging (RabbitMQ, ActiveMQ) works for small scale but breaks at:
- Millions of messages per second
- Multiple consumers reading the same data independently
- Replaying historical data
- Guaranteed ordering

Kafka solves all of these with a **distributed commit log** design.

## Core Architecture

```
                         KAFKA CLUSTER
┌──────────────────────────────────────────────────────────────┐
│                                                              │
│  BROKER 0 (Leader for P0)    BROKER 1 (Leader for P1)       │
│  ┌────────────────────┐      ┌────────────────────┐         │
│  │ Topic: orders       │      │ Topic: orders       │         │
│  │ ┌──────────────┐   │      │ ┌──────────────┐   │         │
│  │ │ Partition 0  │◄──│──┐   │ │ Partition 1  │◄──│──┐      │
│  │ │ (Leader)     │   │  │   │ │ (Leader)     │   │  │      │
│  │ │ Offset: 0-99 │   │  │   │ │ Offset: 0-75 │   │  │      │
│  │ └──────────────┘   │  │   │ └──────────────┘   │  │      │
│  │ ┌──────────────┐   │  │   │ ┌──────────────┐   │  │      │
│  │ │ Partition 1  │   │  │   │ │ Partition 0  │   │  │      │
│  │ │ (Replica)    │   │  │   │ │ (Replica)    │   │  │      │
│  │ └──────────────┘   │  │   │ └──────────────┘   │  │      │
│  └────────────────────┘  │   └────────────────────┘  │      │
│                          │                            │      │
│  BROKER 2 (Leader for P2)│                            │      │
│  ┌────────────────────┐  │                            │      │
│  │ ┌──────────────┐   │  │   Producers write to      │      │
│  │ │ Partition 2  │◄──│──┘   partition leaders        │      │
│  │ │ (Leader)     │   │      Consumers read from      │      │
│  │ └──────────────┘   │      any replica              │      │
│  └────────────────────┘                               │      │
│                                                       │      │
│  ZooKeeper / KRaft ──── Metadata & Leader Election    │      │
└──────────────────────────────────────────────────────────────┘
         ▲                                    │
         │                                    ▼
    PRODUCERS                            CONSUMERS
    (Web apps, CDC,                      (Spark, Flink,
     IoT sensors)                         microservices)
```

## Key Concepts Table

| Concept | Description | Analogy |
|---------|-------------|---------|
| **Broker** | Kafka server storing partitions | A library branch |
| **Topic** | Named stream (category of messages) | A book series (e.g., "orders") |
| **Partition** | Ordered, immutable log within a topic | A volume in the series |
| **Offset** | Position of message in partition | Page number |
| **Producer** | Writes messages to topics | Author |
| **Consumer** | Reads messages from topics | Reader |
| **Consumer Group** | Set of consumers sharing partition reads | Book club (each member reads different chapters) |
| **Replica** | Copy of partition on another broker | Backup copy in another branch |
| **Leader** | Replica that handles reads/writes | Primary copy |
| **ISR** | In-Sync Replicas (up-to-date copies) | Copies that are current |

## Why Partitions Matter

```
Topic "orders" with 6 partitions:

  P0: [msg0, msg1, msg2, ...]  ← Customer A, D (hash("A") % 6 = 0)
  P1: [msg0, msg1, msg2, ...]  ← Customer B, E
  P2: [msg0, msg1, msg2, ...]  ← Customer C, F
  P3: [msg0, msg1, ...]
  P4: [msg0, msg1, ...]
  P5: [msg0, msg1, ...]

Benefits:
  - Parallelism: 6 consumers can read simultaneously
  - Ordering: Messages for same customer are ordered (same partition)
  - Scalability: Add partitions to scale throughput
  - Distribution: Partitions spread across brokers
```

In [ ]:
# Kafka architecture simulation
from collections import defaultdict
import hashlib
import time

print("KAFKA ARCHITECTURE -- Simulated")
print("=" * 60)

class KafkaPartition:
    """Represents a single Kafka partition (ordered log)."""
    def __init__(self, topic, partition_id):
        self.topic = topic
        self.partition_id = partition_id
        self.messages = []  # The log
        self.high_watermark = 0

    def append(self, key, value, timestamp):
        offset = len(self.messages)
        self.messages.append({
            "offset": offset,
            "key": key,
            "value": value,
            "timestamp": timestamp,
        })
        self.high_watermark = offset + 1
        return offset

    def read(self, start_offset, max_records=10):
        return self.messages[start_offset:start_offset + max_records]


class KafkaTopic:
    """Simulates a Kafka topic with multiple partitions."""
    def __init__(self, name, num_partitions=6, replication_factor=3):
        self.name = name
        self.num_partitions = num_partitions
        self.replication_factor = replication_factor
        self.partitions = {i: KafkaPartition(name, i) for i in range(num_partitions)}

    def get_partition(self, key):
        """Determine partition for a key (consistent hashing)."""
        if key is None:
            return hash(time.time()) % self.num_partitions  # Round-robin for null keys
        key_bytes = key.encode() if isinstance(key, str) else str(key).encode()
        return int(hashlib.md5(key_bytes).hexdigest(), 16) % self.num_partitions


class KafkaProducer:
    """Simulates a Kafka producer."""
    def __init__(self, acks="all"):
        self.acks = acks
        self.messages_sent = 0

    def send(self, topic: KafkaTopic, key, value):
        partition_id = topic.get_partition(key)
        partition = topic.partitions[partition_id]
        offset = partition.append(key, value, time.time())
        self.messages_sent += 1
        return partition_id, offset


class KafkaConsumer:
    """Simulates a Kafka consumer in a consumer group."""
    def __init__(self, group_id, assigned_partitions):
        self.group_id = group_id
        self.assigned_partitions = assigned_partitions
        self.committed_offsets = {p: 0 for p in assigned_partitions}

    def poll(self, topic: KafkaTopic, max_records=5):
        records = []
        for pid in self.assigned_partitions:
            start = self.committed_offsets[pid]
            msgs = topic.partitions[pid].read(start, max_records)
            records.extend(msgs)
            self.committed_offsets[pid] = start + len(msgs)
        return records


# Demo: End-to-end Kafka flow
topic = KafkaTopic("orders_events", num_partitions=4)
producer = KafkaProducer(acks="all")

# Produce events
print("  PRODUCING EVENTS:")
events = [
    ("customer_001", {"event": "order_placed", "amount": 150.00}),
    ("customer_002", {"event": "order_placed", "amount": 299.99}),
    ("customer_001", {"event": "payment_received", "amount": 150.00}),
    ("customer_003", {"event": "order_placed", "amount": 75.50}),
    ("customer_002", {"event": "order_shipped", "amount": 299.99}),
    ("customer_001", {"event": "order_shipped", "amount": 150.00}),
    ("customer_003", {"event": "payment_received", "amount": 75.50}),
    ("customer_004", {"event": "order_placed", "amount": 500.00}),
]

for key, value in events:
    pid, offset = producer.send(topic, key, value)
    print(f"    {key} -> Partition {pid}, Offset {offset}: {value['event']}")

# Show partition distribution
print(f"\n  PARTITION DISTRIBUTION:")
for pid, partition in topic.partitions.items():
    keys = set(m["key"] for m in partition.messages)
    print(f"    Partition {pid}: {len(partition.messages)} msgs, keys: {keys if keys else '(empty)'}")

# Consumer group with 2 consumers (each reads 2 partitions)
print(f"\n  CONSUMING (consumer group 'etl_pipeline', 2 consumers):")
consumer_1 = KafkaConsumer("etl_pipeline", [0, 1])
consumer_2 = KafkaConsumer("etl_pipeline", [2, 3])

for i, consumer in enumerate([consumer_1, consumer_2], 1):
    records = consumer.poll(topic)
    print(f"    Consumer {i} (partitions {consumer.assigned_partitions}): {len(records)} records")
    for r in records[:3]:
        print(f"      offset={r['offset']}: {r['key']} - {r['value']['event']}")

print(f"\n  KEY INSIGHT: Same customer ALWAYS goes to same partition = ordered!")
print(f"  Multiple consumers in a group = parallelism without duplication!")

---
# Section 2: Kafka Producer — Writing Data Correctly

## Producer Configuration That Matters

| Config | Description | Recommended |
|--------|-------------|-------------|
| `acks` | How many replicas must confirm | `all` (safest) or `1` (faster) |
| `retries` | Auto-retry on failure | `Integer.MAX_VALUE` (with idempotence) |
| `enable.idempotence` | Prevent duplicate writes on retry | `true` |
| `batch.size` | Bytes to batch before sending | `16384` (16KB) - `65536` (64KB) |
| `linger.ms` | Wait time to fill batch | `5-100` ms |
| `compression.type` | Compress batches | `lz4` or `snappy` |
| `max.in.flight.requests.per.connection` | Concurrent requests | `5` (with idempotence) |

## Partitioning Strategies

| Strategy | How It Works | Use When |
|----------|-------------|----------|
| **Key-based (default)** | `hash(key) % partitions` | Need ordering per entity |
| **Round-robin** | Distribute evenly (null key) | Don't need ordering, max throughput |
| **Custom partitioner** | Your logic decides | Geographic routing, priority |
| **Sticky** | Batch to same partition | No key, want batching efficiency |

## Delivery Guarantees

```
acks=0:   "Fire and forget" -- Producer doesn't wait for confirmation
          Fastest, but may lose messages.

acks=1:   Leader confirms -- Producer waits for leader write
          Good balance. May lose if leader crashes before replication.

acks=all: ALL ISR replicas confirm -- Safest
          Slowest, but no data loss (combined with min.insync.replicas=2)
```

## Exactly-Once Semantics (EOS)

```python
# Enable idempotent producer (prevents duplicates on retry)
producer = KafkaProducer(
    enable_idempotence=True,   # Deduplicates at broker level
    acks='all',                # Wait for all replicas
    retries=MAX_INT,           # Retry indefinitely
    max_in_flight_requests_per_connection=5  # Safe with idempotence
)

# For transactions (atomic writes across topics/partitions):
producer.init_transactions()
producer.begin_transaction()
producer.send(topic1, key, value1)
producer.send(topic2, key, value2)
producer.commit_transaction()  # Atomic: both succeed or both fail
```

In [ ]:
# Producer patterns for data engineering
print("KAFKA PRODUCER PATTERNS FOR DE")
print("=" * 60)

# Pattern 1: CDC Producer (Change Data Capture)
print("""
1. CDC PRODUCER (Debezium pattern):
   - Source: PostgreSQL WAL (Write-Ahead Log)
   - Key: table_name + primary_key (e.g., "orders.12345")
   - Value: {"before": {...}, "after": {...}, "op": "u"}
   - Partition key: primary key (ordering per record)

   Config:
     acks=all, enable.idempotence=true
     key.serializer=StringSerializer
     value.serializer=JsonSerializer (or Avro with Schema Registry)

2. EVENT PRODUCER (Application events):
   - Key: entity_id (user_id, order_id)
   - Value: event payload
   - Ensures ordering per entity

   Config:
     linger.ms=20 (batch for throughput)
     batch.size=65536 (64KB batches)
     compression.type=lz4

3. IOT PRODUCER (Sensor data):
   - Key: device_id
   - Value: {timestamp, readings...}
   - High throughput, tolerates some loss

   Config:
     acks=1 (leader only -- speed over durability)
     batch.size=131072 (128KB)
     linger.ms=100 (aggressive batching)

4. LOG PRODUCER (Application logs):
   - Key: null (round-robin for even distribution)
   - Value: log line (JSON structured)
   - No ordering needed, maximize throughput

   Config:
     acks=0 or 1 (logs are not critical)
     compression.type=snappy
""")

print("  ANTI-PATTERNS:")
print("  - Large messages (>1MB): Use S3 + send reference in Kafka")
print("  - Null keys for ordered data: Breaks ordering guarantee!")
print("  - Too many partitions (>100 per topic): ZooKeeper overhead")
print("  - Not using compression: 2-5x more network/storage cost")

---
# Section 3: Kafka Consumer — Reading Data Correctly

## Consumer Groups: The Scalability Mechanism

```
Topic "orders" (6 partitions):

Consumer Group "spark_streaming" (3 consumers):
  Consumer 1 → reads P0, P1
  Consumer 2 → reads P2, P3
  Consumer 3 → reads P4, P5

Consumer Group "alerting" (2 consumers):
  Consumer A → reads P0, P1, P2
  Consumer B → reads P3, P4, P5

Consumer Group "audit_log" (1 consumer):
  Consumer X → reads P0, P1, P2, P3, P4, P5

KEY RULES:
  - Each partition is read by EXACTLY ONE consumer in a group
  - More consumers than partitions = idle consumers
  - Multiple groups read INDEPENDENTLY (each gets ALL messages)
  - Adding consumers triggers REBALANCE (reassign partitions)
```

## Offset Management

```
Partition 0: [msg0, msg1, msg2, msg3, msg4, msg5, msg6, msg7, msg8, msg9]
                                        ^                              ^
                                   committed                    high watermark
                                   offset=4                     (latest available)

  - Committed offset: "I have processed up to here"
  - On restart, consumer resumes from committed offset
  - auto.commit: risky (may lose or duplicate on crash)
  - manual commit: safe (commit AFTER processing)
```

## Consumer Configuration

| Config | Description | Recommended |
|--------|-------------|-------------|
| `group.id` | Consumer group name | Meaningful name (e.g., `spark_orders_etl`) |
| `auto.offset.reset` | Where to start if no offset | `earliest` (for ETL) or `latest` (for real-time) |
| `enable.auto.commit` | Auto-commit offsets | `false` (manual for exactly-once) |
| `max.poll.records` | Records per poll | `500-1000` |
| `session.timeout.ms` | Detect dead consumer | `30000` (30s) |
| `max.poll.interval.ms` | Max time between polls | `300000` (5min) for heavy processing |

In [ ]:
# Consumer patterns
print("KAFKA CONSUMER PATTERNS")
print("=" * 60)

print("""
1. AT-LEAST-ONCE PROCESSING (most common for ETL):
   while True:
       records = consumer.poll(timeout=1000)
       for record in records:
           process(record)           # Process first
       consumer.commit()             # Then commit
   # If crash after process but before commit:
   # Records will be re-processed (duplicates possible)
   # Fix: Make processing IDEMPOTENT (MERGE/upsert)

2. AT-MOST-ONCE PROCESSING (fire and forget):
   while True:
       records = consumer.poll(timeout=1000)
       consumer.commit()             # Commit FIRST
       for record in records:
           process(record)           # Then process
   # If crash after commit but before process:
   # Records are LOST (already committed)

3. EXACTLY-ONCE PROCESSING (with transactions):
   # Kafka Streams or Flink handle this internally
   # For Spark: use checkpointing + idempotent writes (Delta MERGE)

4. BATCH CONSUMER (for periodic ETL):
   # Read all available records, process batch, commit
   consumer.assign(partitions)       # Manual assignment
   consumer.seek_to_beginning()      # Or specific offset
   records = []
   while has_more:
       batch = consumer.poll(timeout=5000)
       records.extend(batch)
   process_batch(records)
   consumer.commit()

5. SPARK STRUCTURED STREAMING (recommended for DE):
   df = (spark.readStream
       .format("kafka")
       .option("kafka.bootstrap.servers", "broker1:9092")
       .option("subscribe", "orders")
       .option("startingOffsets", "earliest")
       .load())
   # Spark manages offsets via checkpointing (exactly-once with Delta)
""")

---
# Section 4: Schema Registry, Kafka Connect & Operations

## Schema Registry

Problem: Producers and consumers must agree on message format.
Without schema management, schema changes break consumers.

```
Producer → Schema Registry → Broker → Consumer
              ↕                          ↕
         Validates schema            Fetches schema
         Assigns schema ID           Deserializes

Schema Evolution Rules:
  BACKWARD:  New schema can read old data (add optional fields)
  FORWARD:   Old schema can read new data (remove optional fields)
  FULL:      Both directions (safest, most restrictive)
```

## Kafka Connect

Pre-built connectors for common data sources and sinks:

| Connector | Direction | Use Case |
|-----------|-----------|----------|
| JDBC Source | DB → Kafka | Ingest from PostgreSQL, MySQL |
| Debezium | DB WAL → Kafka | CDC (Change Data Capture) |
| S3 Sink | Kafka → S3 | Archive to data lake |
| Elasticsearch Sink | Kafka → ES | Search indexing |
| BigQuery Sink | Kafka → BQ | Analytics |
| HDFS Sink | Kafka → HDFS | Legacy lake ingestion |

## Operations: Key Metrics to Monitor

| Metric | Alert Threshold | Meaning |
|--------|----------------|---------|
| Consumer lag | > 10,000 messages | Consumer falling behind |
| Under-replicated partitions | > 0 | Broker health issue |
| ISR shrink rate | > 0 | Replicas falling behind |
| Request latency (p99) | > 100ms | Broker overloaded |
| Disk usage | > 80% | Need retention cleanup |
| Active controller count | != 1 | Leadership election issue |

## Retention & Compaction

```
TIME-BASED RETENTION (default):
  retention.ms=604800000 (7 days)
  Messages older than 7 days are deleted
  Good for: Event streams, logs

LOG COMPACTION:
  cleanup.policy=compact
  Keeps ONLY latest value per key (like a table)
  Good for: CDC (latest state per record), config updates
  
  Before compaction: key=A:v1, key=B:v1, key=A:v2, key=B:v2, key=A:v3
  After compaction:  key=A:v3, key=B:v2  (only latest per key)
```

In [ ]:
# Kafka + Spark Structured Streaming integration
print("KAFKA + SPARK STRUCTURED STREAMING")
print("=" * 60)

print("""
# READING FROM KAFKA (Structured Streaming):
raw_stream = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092,broker2:9092")
    .option("subscribe", "orders,payments")   # Multiple topics
    .option("startingOffsets", "earliest")     # Or "latest"
    .option("maxOffsetsPerTrigger", 100000)    # Rate limit
    .option("kafka.security.protocol", "SASL_SSL")  # Production auth
    .load()
)

# Kafka message schema:
# key (binary), value (binary), topic, partition, offset, timestamp

# PARSING (value is binary -- deserialize to JSON):
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType

order_schema = StructType([...])  # Define your schema

parsed = (raw_stream
    .selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "timestamp")
    .withColumn("data", from_json(col("value"), order_schema))
    .select("key", "timestamp", "data.*")
)

# WRITING TO DELTA (exactly-once with checkpointing):
query = (parsed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/checkpoints/orders_stream")
    .trigger(processingTime="30 seconds")  # Or availableNow=True for batch
    .toTable("bronze.orders_raw")
)

# WRITING BACK TO KAFKA:
output_stream = (transformed.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("topic", "orders_enriched")
    .option("checkpointLocation", "/checkpoints/orders_enriched")
    .start()
)
""")

print("  Key configurations for production:")
print("  - checkpointLocation: REQUIRED for exactly-once")
print("  - maxOffsetsPerTrigger: Rate-limit to prevent OOM")
print("  - trigger(availableNow=True): Batch mode on streaming source")
print("  - failOnDataLoss=false: Don't fail if offsets are expired")

---
# Section 5: Kafka Interview Questions

In [ ]:
print("KAFKA INTERVIEW QUESTIONS")
print("=" * 60)

questions = [
    {
        "q": "How does Kafka guarantee ordering?",
        "a": "Ordering is guaranteed WITHIN a partition only. Same key always routes to same partition via hash(key) % num_partitions. For global ordering, use a single partition (sacrifices throughput)."
    },
    {
        "q": "What happens when a consumer in a group crashes?",
        "a": "After session.timeout.ms (default 30s), the group coordinator triggers a REBALANCE. Crashed consumer's partitions are reassigned to remaining consumers. During rebalance, consumption pauses briefly."
    },
    {
        "q": "Explain consumer lag and why it matters.",
        "a": "Consumer lag = difference between latest offset produced and latest offset committed by consumer. High lag means consumers can't keep up. Causes: slow processing, insufficient consumers, or producer spike. Fix: add consumers, optimize processing, or scale partitions."
    },
    {
        "q": "How would you handle exactly-once processing with Kafka?",
        "a": "Three approaches: (1) Idempotent producer + transactional consumer (Kafka native), (2) Spark Structured Streaming with checkpointing + Delta Lake MERGE (DE approach), (3) Consumer-side deduplication using a lookup table of processed offsets."
    },
    {
        "q": "When would you use log compaction vs time-based retention?",
        "a": "Time-based: event streams, logs, metrics (all events matter). Compaction: CDC topics (only latest state per key matters), configuration topics, materialized views. Compaction keeps only the LATEST value per key, acting like a changelog table."
    },
    {
        "q": "How do you scale Kafka consumers?",
        "a": "Add consumers to the consumer group (up to # partitions). Beyond that, increase partitions (cannot decrease!). Rule of thumb: partitions >= max expected consumers. For Spark: increase streaming micro-batch parallelism."
    },
    {
        "q": "What is the difference between Kafka and a traditional message queue?",
        "a": "Traditional queues (RabbitMQ): message consumed = message deleted. Kafka: messages RETAINED (configurable duration). Multiple consumer groups read independently. Kafka supports replay (seek to offset). Kafka scales horizontally via partitions."
    },
]

for i, qa in enumerate(questions, 1):
    print(f"\nQ{i}: {qa['q']}")
    print(f"{'─'*60}")
    print(f"  {qa['a']}")

---
# Summary: Kafka Cheat Sheet

## Architecture Quick Reference
- **Broker**: Server storing partitions
- **Topic**: Named stream (logical grouping)
- **Partition**: Ordered log (parallelism + ordering unit)
- **Consumer Group**: Load balancing mechanism
- **Offset**: Message position (consumer's progress bookmark)

## When to Use Kafka

| Use Case | Kafka Pattern |
|----------|--------------|
| CDC (database changes) | Debezium → Kafka → Delta Lake |
| Event streaming | App → Kafka → Spark Streaming |
| Log aggregation | Apps → Kafka → Elasticsearch |
| Microservice communication | Service A → Kafka → Service B |
| Real-time analytics | Kafka → Flink/Spark → Dashboard |
| Data pipeline buffer | Source → Kafka → ETL → Warehouse |

## Kafka vs Alternatives

| Feature | Kafka | RabbitMQ | AWS Kinesis | Pulsar |
|---------|-------|----------|-------------|--------|
| Throughput | Very high | Medium | High | Very high |
| Retention | Configurable (days/forever) | Until consumed | 7 days max | Configurable |
| Ordering | Per partition | Per queue | Per shard | Per partition |
| Replay | Yes (seek to offset) | No | Limited | Yes |
| Managed options | Confluent Cloud, MSK | CloudAMQP | Native AWS | StreamNative |

---
*Apache Kafka Deep Dive for Data Engineers*